# 🛡️ Branch Protection, Required Checks & CI Quality Gates

> **Every section answers four questions: WHY this exists, WHAT it is, HOW it works, WHEN to use it.**
> Real-world scenarios, ❌ before / ✅ after patterns, and *"How to configure this in GitHub"* walkthroughs.

---

**Topics**
1. Branch Protection — The Gatekeeper Mental Model
2. Required Status Checks — The Automated Quality Contract
3. CODEOWNERS — Automatic Review Assignment
4. Ruff — The All-in-One Python Linter
5. Isort via Ruff — Import Order as a Quality Signal
6. Protection Tests — Structural Repo Health Checks
7. Configuring Required Checks in GitHub (step-by-step)
8. The Full CI Quality Stack — How It All Connects
9. Real-World: How Google and GitHub Enforce Quality at Scale
10. Senior Architect Interview Questions

---
## 1 · Branch Protection — The Gatekeeper Mental Model

### 🧠 Mental Model — *The Airlock*

> **Branch protection is the airlock between your team's work and your shared `main` branch. Just as a spacecraft airlock prevents vacuum from reaching the cabin, branch protection prevents bad code from reaching the branch that drives your deployments. The airlock can be configured: some checks are advisory (you can override), some are mandatory (no override possible).**

**WHY branch protection exists:** Without it:
- Any engineer with push access can commit directly to `main` — bypassing all CI checks
- A tired engineer at 4 PM can push broken code that triggers a production deploy
- `git push --force` can rewrite history, losing commits others depended on
- There is no review gate — code is reviewed by no one before reaching production

**WHAT it is:** A set of rules GitHub enforces on a specific branch (usually `main`). Rules can require:
- Pull requests (no direct push)
- A minimum number of approving reviews before merge
- Specific CI status checks to pass (the `required status checks`)
- CODEOWNER review when code owners' files are changed
- Signed commits (GPG/SSH verification)
- Linear history (no merge commits — squash or rebase only)

**HOW it works:** GitHub's API enforces these rules server-side. Even a repo admin with local `git push --force` access is subject to these rules if "Include administrators" is checked. The rules live in the branch protection configuration (Settings → Branches).

**WHEN to add which rules:**

| Rule | Add when | Risk if missing |
|---|---|---|
| Require PR | From day 1 (even solo projects) | Direct pushes bypass all CI |
| Require 1 review | 2+ engineers | Unreviewed code reaches prod |
| Required checks | First CI job exists | CI can be bypassed by pushing directly |
| Dismiss stale reviews | Important security code | Old approval valid after new, unreviewed commits |
| CODEOWNER review | Security/infra files changed | CI config changed without senior review |
| Linear history | Consistent git log | Messy merge commits in history |

---
## 2 · Required Status Checks — The Automated Quality Contract

### 🧠 Mental Model — *The Pull-Request Contract*

> **A required status check is a promise: this specific automated check must report green before any PR can merge. It is a contract between the team and the codebase. By making checks required, you remove the social pressure to merge anyway. The check is not a guideline — it is a gate. The gate cannot be argued away. This is the difference between "we try to always run tests" and "it is impossible to merge without passing tests."**

### How Required Checks Work

```
GitHub Flow with Required Checks:

1. Engineer opens PR from feature/payment-idempotency → main
   
2. GitHub sees the PR and triggers configured workflows:
   CI workflow: lint job → running ⏳
   CI workflow: test job → running ⏳
   CI workflow: isort job → running ⏳
   CI workflow: secret-scan job → running ⏳

3. Branch protection check:
   REQUIRED: lint       → ✅ passed
   REQUIRED: test       → ✅ passed
   REQUIRED: isort      → ❌ FAILED (import order violation)
   REQUIRED: secret-scan → ✅ passed

4. GitHub BLOCKS the merge button:
   "Merging is blocked: Required status check 'isort' is failing"
   Even repo admins cannot merge (if 'Include administrators' is ON)

5. Engineer fixes import order locally:
   ruff check --select I --fix .
   git add . && git commit -m 'fix: import order'
   git push

6. CI reruns → all required checks pass → merge button unblocked ✅
```

### Configuring Required Checks — Step by Step

**In GitHub UI (Settings → Branches → Add rule):**

```
Branch name pattern:  main

✅ Require a pull request before merging
   Required approving reviews: 1
   ✅ Dismiss stale pull request approvals when new commits are pushed
   ✅ Require review from Code Owners

✅ Require status checks to pass before merging
   ✅ Require branches to be up to date before merging
   
   Status checks found in the last week for this repository:
   [Search: lint]     → Add: "Lint & Format (ruff)"
   [Search: test]     → Add: "Protection Tests (pytest)"
   [Search: isort]    → Add: "Import Order (isort via ruff)"
   [Search: yaml]     → Add: "YAML Lint (workflows)"
   [Search: secret]   → Add: "Secret Detection"

✅ Require linear history (squash/rebase merges only)
✅ Include administrators (NO escape hatch for admins)
✅ Restrict who can push to matching branches: [karthikb35]
```

### ⚠️ The "Strict" vs "Loose" Required Checks Trade-off

```
"Require branches to be up to date" = STRICT mode

PRO: The branch is tested against the latest main before merging.
     Two PRs that both pass CI independently can still break together.
     Strict mode catches the interaction.

CON: In a busy repo with many simultaneous PRs, strict mode creates a
     'merge queue' problem: PR A merges, PR B must now rebase and re-run CI,
     then PR C must rebase and re-run CI. Throughput degrades quadratically.

SOLUTION: GitHub Merge Queue (available for GitHub Enterprise / Teams):
  PRs enter a queue; GitHub merges them serially with CI running on the
  combined state. Each PR's CI runs once, against the final merged state.
```

In [ ]:
"""
Branch Protection Rule Validator
=================================
Queries the GitHub API to check if a repository's branch protection
matches a minimum-security configuration.

WHY: Branch protection rules can drift. An admin disables a required check
"temporarily" to unblock a hotfix and never re-enables it. This tool
detects that drift in CI.

Usage:
    GITHUB_TOKEN=ghp_xxx python check_branch_protection.py owner/repo main
"""
from __future__ import annotations

import os
import sys
from dataclasses import dataclass, field
from typing import Any


@dataclass
class BranchProtectionAudit:
    branch: str
    required_pr: bool = False
    min_reviews: int = 0
    dismiss_stale: bool = False
    require_codeowner: bool = False
    required_checks: list[str] = field(default_factory=list)
    include_admins: bool = False
    linear_history: bool = False

    def violations(self, policy: "BranchPolicy") -> list[str]:
        issues = []
        if policy.require_pr and not self.required_pr:
            issues.append("PRs not required — direct push to main is possible")
        if self.min_reviews < policy.min_reviews:
            issues.append(f"Only {self.min_reviews} review(s) required; policy needs {policy.min_reviews}")
        if policy.dismiss_stale and not self.dismiss_stale:
            issues.append("Stale approvals not dismissed — old approval still valid after new commits")
        if policy.require_codeowner and not self.require_codeowner:
            issues.append("CODEOWNER review not required")
        if not self.include_admins:
            issues.append("Admins can bypass protection — include_administrators should be True")
        missing = set(policy.required_checks) - set(self.required_checks)
        if missing:
            issues.append(f"Missing required checks: {sorted(missing)}")
        return issues


@dataclass
class BranchPolicy:
    """Minimum required configuration for the main branch."""
    require_pr:        bool = True
    min_reviews:       int  = 1
    dismiss_stale:     bool = True
    require_codeowner: bool = True
    required_checks:   list[str] = field(default_factory=lambda: [
        "Lint & Format (ruff)",
        "Import Order (isort via ruff)",
        "Protection Tests (pytest)",
        "YAML Lint (workflows)",
        "Secret Detection",
    ])


# Simulate two scenarios: compliant and non-compliant
policy = BranchPolicy()

compliant = BranchProtectionAudit(
    branch="main",
    required_pr=True,
    min_reviews=1,
    dismiss_stale=True,
    require_codeowner=True,
    required_checks=[
        "Lint & Format (ruff)",
        "Import Order (isort via ruff)",
        "Protection Tests (pytest)",
        "YAML Lint (workflows)",
        "Secret Detection",
    ],
    include_admins=True,
    linear_history=True,
)

drifted = BranchProtectionAudit(
    branch="main",
    required_pr=True,
    min_reviews=0,               # someone removed review requirement
    dismiss_stale=False,         # stale approvals not dismissed
    require_codeowner=False,     # CODEOWNER review disabled
    required_checks=[
        "Lint & Format (ruff)",  # missing 4 required checks
    ],
    include_admins=False,        # admins can bypass
    linear_history=False,
)

print("=== Branch Protection Audit ===")
for name, audit in [("Compliant config", compliant), ("Drifted config", drifted)]:
    violations = audit.violations(policy)
    status = "✅ PASS" if not violations else f"❌ {len(violations)} VIOLATION(S)"
    print(f"\n{name}: {status}")
    for v in violations:
        print(f"  ✗  {v}")

---
## 3 · CODEOWNERS — Automatic Review Assignment

### 🧠 Mental Model — *The Expert Directory*

> **CODEOWNERS is a file that maps file patterns to GitHub users or teams. When a PR changes a file that matches a pattern, the mapped owner is automatically added as a required reviewer. It turns implicit knowledge ("always ask Karthik before changing the CI config") into explicit, automated enforcement. Nobody forgets to tag the right reviewer, because GitHub does it automatically.**

**WHY CODEOWNERS exists:**
- Large codebases have domain experts — the infra team owns `.github/`, the frontend team owns `src/ui/`
- Without CODEOWNERS, a new engineer can merge changes to the CI workflow without any expert reviewing it
- A compromised CI workflow can exfiltrate secrets, ship malicious code, or disable security checks
- CODEOWNERS ensures that high-impact files always get reviewed by the right person

### The CODEOWNERS Syntax

```
# File: .github/CODEOWNERS

# Global owner — all files (evaluated last = lowest specificity)
*                       @karthikb35

# Specific patterns override the global rule (last match wins)
.github/                @karthikb35 @security-team
study-guide/            @karthikb35
*.tf                    @karthikb35 @infrastructure-team
kubernetes/             @karthikb35 @platform-team

# Team ownership (team must be in the same org)
src/payments/           @my-org/payments-team

# Files without a matching pattern → no required reviewer
# (PR can merge with just the minimum review count)
```

### Pairing CODEOWNERS with Branch Protection

```
CODEOWNERS alone: owner is automatically REQUESTED for review, but not REQUIRED.

CODEOWNERS + branch protection:
  Settings → Branches → main → ✅ "Require review from Code Owners"
  
  Now: if a PR touches a CODEOWNER-mapped file AND the owner hasn't approved,
  the merge is BLOCKED — not just soft-requested.

This is the security model GitHub itself uses for its own repositories.
Critical security files require sign-off from the security team.
No PR can ship a CI change without the platform team's approval.
```

### 🌍 Real-World: Google's OWNERS Files
Google's monorepo uses a similar mechanism called OWNERS files. Every directory has an OWNERS file listing who can approve changes in that directory. The key rules:
1. **Nearest OWNERS file wins** — the file closest to the changed path takes precedence
2. **Inherited ownership** — if no OWNERS file in a directory, it inherits from the parent
3. **Automated tooling** — Gerrit (their code review system) automatically assigns reviewers based on OWNERS
4. **No PR merges without approval from at least one person in OWNERS** — technically enforced

This ensures that at scale (25,000 engineers, 2B+ lines of code), the right expert always reviews changes to their area of ownership — automatically, without any human coordination.

---
## 4 · Ruff — The All-in-One Python Linter

### 🧠 Mental Model — *The Compiler for Code Quality*

> **Ruff is to Python linting what a compiler is to compilation: a single, fast, authoritative tool that catches a class of problems before they reach production. Pre-ruff, a Python project needed 5+ separate tools (flake8, pycodestyle, isort, pyupgrade, bandit). Ruff replaces all of them with one tool that runs in milliseconds — removing the excuse "I didn't run the linter because it was too slow."**

**WHY ruff over flake8/isort/pylint:**
- **Speed:** 10–100× faster than equivalents (written in Rust). A 100k-line codebase lints in ~200ms instead of 20+ seconds.
- **Consolidation:** One config file (`ruff.toml`), one tool, one CI step
- **Auto-fix:** `ruff check --fix .` fixes most issues automatically
- **Editor integration:** First-class VS Code extension, LSP protocol
- **Formatter:** `ruff format` replaces Black (same algorithm, faster)

### Ruff Rule Categories

| Category | Code | What it catches | Example |
|---|---|---|---|
| pyflakes | F | Undefined names, unused imports | `import os` never used |
| pycodestyle | E, W | Style violations | Missing whitespace, line too long |
| isort | I | Import order | stdlib before third-party |
| pyupgrade | UP | Old Python syntax | `dict()` → `{}` |
| flake8-bugbear | B | Likely bugs | Mutable default arg, `assert` in tests |
| flake8-bandit | S | Security issues | `exec()`, `subprocess` without check |
| flake8-simplify | SIM | Overly complex code | `if x == True:` → `if x:` |
| pep8-naming | N | Naming conventions | `def MyFunction()` → `def my_function()` |

### The Before/After: Setting Up Ruff in a Project

```
❌ BEFORE: 5 tools, 5 configs, 5 CI steps, 20+ seconds to run
  .flake8         → flake8 configuration
  setup.cfg       → isort configuration
  pyproject.toml  → black + pylint + bandit
  .pre-commit-config.yaml → 5 hooks
  CI time: 22 seconds

✅ AFTER: 1 tool, 1 config, 1 CI step, 200ms to run
  ruff.toml       → everything
  CI time: 0.2 seconds
```

### 🌍 Where Ruff is Used in Production
- **FastAPI / Starlette:** migrated from flake8 to ruff in 2023
- **Pydantic v2:** uses ruff for all CI linting
- **Astral (ruff's creator):** dogfoods ruff on ruff itself
- **Hugging Face transformers:** 260k+ lines, linted in ~400ms with ruff
- **Our repo:** `ruff.toml` at repo root; CI requires `ruff check` and `ruff format --check` to pass

In [ ]:
"""
Ruff Rule Demonstrator
======================
Shows the categories of issues ruff catches, with real examples.
Run: ruff check --select ALL demo_ruff_issues.py
     to see these violations flagged.
"""
from __future__ import annotations

import json
import os
from dataclasses import dataclass
from typing import Any


# ── Simulated violations (intentional, for teaching purposes) ──────────
# In the real codebase, ruff catches these before they're committed.

VIOLATIONS = [
    {
        "rule": "F401 (pyflakes)",
        "description": "Unused import",
        "before": "import os\nimport json  # never used",
        "after": "import os",
        "auto_fixable": True,
    },
    {
        "rule": "I001 (isort)",
        "description": "Import order violation",
        "before": "import json\nimport os\nfrom pathlib import Path",  # wrong order
        "after": "from pathlib import Path\nimport json\nimport os",   # stdlib first, alpha
        "auto_fixable": True,
    },
    {
        "rule": "UP007 (pyupgrade)",
        "description": "Use X | Y instead of Union[X, Y] (Python 3.10+)",
        "before": "from typing import Union\ndef f(x: Union[str, None]) -> None: ...",
        "after": "def f(x: str | None) -> None: ...",
        "auto_fixable": True,
    },
    {
        "rule": "B006 (bugbear)",
        "description": "Mutable default argument — a classic Python bug",
        "before": "def process(items: list = []) -> list: ...",
        "after": "def process(items: list | None = None) -> list:\n    if items is None: items = []",
        "auto_fixable": False,
    },
    {
        "rule": "S603 (bandit)",
        "description": "subprocess call with shell=True — command injection risk",
        "before": "import subprocess\nsubprocess.run(user_input, shell=True)",
        "after": "subprocess.run(['cmd', '--flag'], check=True)  # list, not string",
        "auto_fixable": False,
    },
    {
        "rule": "SIM108 (simplify)",
        "description": "Ternary expression instead of if-else block",
        "before": "if condition:\n    x = 'yes'\nelse:\n    x = 'no'",
        "after": "x = 'yes' if condition else 'no'",
        "auto_fixable": True,
    },
]

print("=== Ruff — What It Catches (and Auto-Fixes) ===")
for v in VIOLATIONS:
    fixable = "✅ auto-fixable" if v["auto_fixable"] else "⚠️  manual fix required"
    print(f"\n[{v['rule']}] {v['description']} — {fixable}")
    print(f"  ❌ Before: {v['before'].split(chr(10))[0]}")
    print(f"  ✅ After:  {v['after'].split(chr(10))[0]}")

auto_fixable = sum(1 for v in VIOLATIONS if v["auto_fixable"])
print(f"\nSummary: {auto_fixable}/{len(VIOLATIONS)} violations auto-fixable by 'ruff check --fix .'")

---
## 5 · The Full CI Quality Stack — How It All Connects

### 🧠 Mental Model — *Concentric Rings of Defence*

> **Each CI check is a ring of defence. The outermost ring (fastest) catches the cheapest-to-fix problems first. The inner rings (slower) catch deeper issues. A problem caught in the outer ring (ruff style violation) costs 2 seconds to fix. The same problem in the inner ring (production outage) costs 2 hours. The concentric ring model makes the cost of defects visible — fix them at the cheapest ring.**

```
Ring 0: Pre-commit hook (local, < 1s)
  ruff check --fix . && ruff format .
  → Catches style, imports, basic security before git commit

Ring 1: PR CI — fast checks (parallel, < 3 min)
  lint: ruff check .             → style + security + imports
  isort: ruff check --select I   → import order (separate required check)
  yaml-lint: yamllint .          → workflow YAML syntax
  secret-scan: trufflehog        → credential leak detection

Ring 2: PR CI — structural tests (parallel with Ring 1, < 5 min)
  test: pytest tests/            → notebook JSON, workflow timeouts,
                                   HTML book structure, CODEOWNERS present

Ring 3: PR review (human)
  CODEOWNER review               → domain expert approval
  Peer review                    → logic, architecture, intent

Ring 4: Deployment gates (post-merge)
  Pages build                    → HTML site renders correctly
  PDF render                     → books render to PDF without errors

Ring 5: Production (post-deploy)
  Synthetic tests                → site is reachable, key pages load
```

### The Required Checks Configuration Matrix

| Check name (exact GitHub name) | CI job name | What it verifies | Blocking? |
|---|---|---|---|
| `Lint & Format (ruff)` | `lint` | Style, security patterns, imports | ✅ Yes |
| `Import Order (isort via ruff)` | `isort` | Import sort order | ✅ Yes |
| `Protection Tests (pytest)` | `test` | Notebook structure, workflow YAML, HTML books | ✅ Yes |
| `YAML Lint (workflows)` | `yaml-lint` | Workflow file syntax | ✅ Yes |
| `Secret Detection` | `secret-scan` | No credentials in code | ✅ Yes |
| `Build Pages site` | `build` (pages.yml) | HTML site assembles correctly | ✅ Yes |
| CODEOWNER approval | — (GitHub native) | Domain expert reviewed it | ✅ Yes |

### 🌍 Real-World: GitHub's Own CI for GitHub
GitHub's engineering team runs 10,000+ CI checks per hour on their own monorepo. Key learnings published in their engineering blog:
1. **Fast checks first:** They separate lint (seconds) from integration tests (minutes) into different required check groups. Fast checks provide immediate feedback; slow checks run in the background.
2. **Required checks are non-negotiable:** Even the CEO's PR must pass all required checks. "Include administrators" is always ON. The check is cultural enforcement via technical mechanism.
3. **Flaky tests are zero tolerance:** A flaky required check that sometimes fails for non-code reasons erodes trust in all required checks. Engineers start ignoring CI output. Fix flaky tests immediately or remove them from required checks.
4. **Merge queues for throughput:** With 200+ PRs per day, strict branch-is-up-to-date requirements created a bottleneck. Merge queues (PRs automatically serialised and tested against the latest main) solved this while maintaining correctness.

In [ ]:
"""
CI Check Health Monitor
========================
Simulates a CI health dashboard that tracks:
- Pass rate per check over time
- Flakiness detection (passes and failures for the same commit)
- Time-to-green for the critical path

This is the kind of internal tool platform teams build to keep
CI healthy and maintain trust in the required checks.
"""
from __future__ import annotations

import random
import statistics
from dataclasses import dataclass, field
from datetime import datetime, timedelta
from enum import Enum


class CheckResult(Enum):
    PASS    = "pass"
    FAIL    = "fail"
    FLAKY   = "flaky"   # passed on re-run → flaky, not a real failure


@dataclass
class CheckRun:
    name: str
    commit_sha: str
    duration_s: float
    result: CheckResult
    timestamp: datetime = field(default_factory=datetime.now)


@dataclass
class CIHealthReport:
    check_name: str
    runs: list[CheckRun]

    @property
    def pass_rate(self) -> float:
        if not self.runs:
            return 0.0
        passing = sum(1 for r in self.runs if r.result == CheckResult.PASS)
        return passing / len(self.runs)

    @property
    def flakiness_rate(self) -> float:
        if not self.runs:
            return 0.0
        flaky = sum(1 for r in self.runs if r.result == CheckResult.FLAKY)
        return flaky / len(self.runs)

    @property
    def p95_duration_s(self) -> float:
        if not self.runs:
            return 0.0
        durations = sorted(r.duration_s for r in self.runs)
        idx = int(len(durations) * 0.95)
        return durations[min(idx, len(durations) - 1)]

    @property
    def health_status(self) -> str:
        if self.pass_rate >= 0.99 and self.flakiness_rate < 0.02:
            return "✅ HEALTHY"
        if self.flakiness_rate >= 0.05:
            return "⚠️  FLAKY — investigate immediately"
        if self.pass_rate < 0.90:
            return "❌ FAILING — blocking team"
        return "⚠️  DEGRADED"


def simulate_check(name: str, base_pass_rate: float, flake_rate: float,
                   avg_duration: float, n_runs: int = 100) -> CIHealthReport:
    random.seed(42)
    runs = []
    for i in range(n_runs):
        sha = f"{random.randint(0, 0xFFFFFF):06x}"
        duration = max(0.5, random.gauss(avg_duration, avg_duration * 0.15))
        r = random.random()
        if r < base_pass_rate - flake_rate:
            result = CheckResult.PASS
        elif r < base_pass_rate:
            result = CheckResult.FLAKY
        else:
            result = CheckResult.FAIL
        runs.append(CheckRun(name=name, commit_sha=sha, duration_s=duration, result=result))
    return CIHealthReport(check_name=name, runs=runs)


checks = [
    simulate_check("lint",        base_pass_rate=0.995, flake_rate=0.002, avg_duration=12.0),
    simulate_check("isort",       base_pass_rate=0.98,  flake_rate=0.001, avg_duration=8.0),
    simulate_check("test",        base_pass_rate=0.96,  flake_rate=0.03,  avg_duration=45.0),
    simulate_check("yaml-lint",   base_pass_rate=0.999, flake_rate=0.000, avg_duration=5.0),
    simulate_check("secret-scan", base_pass_rate=0.997, flake_rate=0.001, avg_duration=25.0),
]

print("=== CI Health Dashboard (last 100 runs) ===")
print(f"{'Check':25s}  {'Pass%':6s}  {'Flaky%':7s}  {'p95 Duration':12s}  Status")
print("─" * 75)
for report in checks:
    print(
        f"{report.check_name:25s}  "
        f"{report.pass_rate:5.1%}   "
        f"{report.flakiness_rate:5.1%}    "
        f"{report.p95_duration_s:8.1f}s     "
        f"{report.health_status}"
    )

print("\n⚠️  ACTION REQUIRED: 'test' check flakiness is 3% — investigate and fix before it erodes trust")
print("   Rule of thumb: >2% flakiness on a required check → remove from required OR fix within 1 sprint")

---
## 6 · Configuring Everything — The Complete Setup Guide

### Step 1: Pre-commit Hook (local developer experience)

```yaml
# .pre-commit-config.yaml — runs before every git commit
repos:
  - repo: https://github.com/astral-sh/ruff-pre-commit
    rev: v0.5.0
    hooks:
      - id: ruff
        args: [--fix]      # auto-fix what can be fixed
      - id: ruff-format    # auto-format

# Install: pip install pre-commit && pre-commit install
# This runs ruff on every git commit — zero overhead for the developer
# because ruff runs in ~200ms even on large codebases.
```

### Step 2: CI Workflow (automated gate on every PR)

Already created in `.github/workflows/ci.yml`. The key: each job has a descriptive name that exactly matches the required status check name configured in GitHub branch protection. **The check name in GitHub Settings must exactly match the job's `name:` field in the YAML.**

```yaml
# In ci.yml:
jobs:
  lint:
    name: Lint & Format (ruff)     ← THIS EXACT STRING
```

```
# In GitHub Settings → Branches → main → Required status checks:
Search: "Lint & Format (ruff)"    ← MUST MATCH EXACTLY
```

### Step 3: GitHub Branch Protection Rules

```
Navigate to: github.com/karthikb35/devops-study/settings/branches

Click: "Add rule" or edit existing main rule

Branch name pattern: main

✅ Require a pull request before merging
   Required approving reviews: 1
   ✅ Dismiss stale pull request approvals when new commits are pushed
   ✅ Require review from Code Owners
   ✅ Require approval of the most recent reviewable push

✅ Require status checks to pass before merging  
   ✅ Require branches to be up to date before merging
   
   Status checks (add each by exact name after first CI run):
   • Lint & Format (ruff)
   • Import Order (isort via ruff)
   • Protection Tests (pytest)
   • YAML Lint (workflows)
   • Secret Detection
   • Build Pages site

✅ Require conversation resolution before merging
✅ Require linear history
✅ Do not allow bypassing the above settings (include administrators)
✅ Restrict who can push to matching branches
   Allowed: karthikb35

Click: Save changes
```

### Step 4: Verify It Works

```bash
# Create a test PR with an intentional violation
git checkout -b test/verify-protection
echo 'import json\nimport os'  > test_import_order.py  # wrong order: json before os
git add test_import_order.py
git commit -m 'test: verify isort check'
git push origin test/verify-protection
# Open PR on GitHub
# Expected: isort check FAILS, merge button is blocked

# Fix it:
ruff check --select I --fix .  # auto-fix import order
git add . && git commit -m 'fix: import order' && git push
# Expected: isort check passes, merge button unblocks
```

---
## 7 · Senior Architect Interview Questions

### Q1. Your team says "we always run tests before merging." What's the problem with this, and what do you do instead?

**Answer:** "We always" is a social contract, not a technical one. Social contracts fail under pressure: a hotfix at 2 AM, a deadline tomorrow, a junior engineer who doesn't know the process. The fix is to make the correct behaviour the only possible behaviour via required status checks. With required checks configured and "Include administrators" enabled, it is technically impossible to merge without passing tests — no matter the time pressure or who is merging. The check cannot be argued away, forgotten, or bypassed. Social contracts are weak; technical enforcement is strong.

### Q2. Describe the CODEOWNERS mechanism and how it differs from just requiring X reviews.

**Answer:** Requiring X reviews ensures that Y people look at the change. CODEOWNERS ensures that the *right* people look at the change. Without CODEOWNERS, a PR that changes the CI workflow might be approved by two backend engineers who know nothing about pipelines. With CODEOWNERS, any PR touching `.github/` automatically requires review from the platform team — technically enforced. CODEOWNERS turns implicit domain expertise into explicit, automated review routing. Combined with `Require review from Code Owners` in branch protection, it becomes a hard gate: no merge until the domain expert approves.

### Q3. Why is ruff better than flake8 + isort + bandit for a CI pipeline?

**Answer:** Three reasons:
1. **Speed:** Ruff is 10-100× faster (Rust vs Python). On a 100k-line codebase: ruff runs in ~200ms; flake8+isort+bandit take 20+ seconds. Fast checks get run; slow checks get skipped under time pressure.
2. **Simplicity:** One tool, one config file, one CI step instead of 3+ tools with overlapping configuration and version conflicts.
3. **Consistency:** All rules interpreted by the same engine. Flake8 plugins can conflict; ruff's rules are unified. Auto-fix works across all rule categories because one tool owns the entire AST.

### Q4. A required status check is flaky — passes 95% of the time. What do you do?

**Answer:** A flaky required check is worse than no check. It creates two problems:
1. Engineers re-run pipelines without investigating why they failed — the check becomes noise
2. Real failures get ignored because "it's probably just flaky"

**Response:** Immediately remove it from required checks (it's not providing reliable signal anyway). Fix the underlying flakiness in a sprint. Add it back as required only after achieving >99% pass rate over 2 weeks. The threshold: >2% flakiness = remove from required. This is the "you build it, you run it" principle applied to CI: the team that writes a check is responsible for its reliability.

### Q5. How do you prevent a compromised or malicious GitHub Action from exfiltrating secrets?

**Answer:** Five layers:
1. **Pin actions to SHA:** `uses: actions/checkout@11bd71901...` (SHA) not `@v4` (mutable tag). A compromised release that updates the `v4` tag doesn't affect you.
2. **Minimum permissions:** `permissions: contents: read` in every workflow; explicitly grant only what's needed per job.
3. **OIDC over stored secrets:** No AWS keys to steal — credentials are ephemeral per job.
4. **CODEOWNERS on `.github/`:** Any PR changing a workflow requires the platform team's review.
5. **`prevent-secrets` on push:** trufflehog/detect-secrets as a required check blocks accidental credential commits.

### Q6. A senior engineer bypasses branch protection with admin override. How do you prevent this architecturally?

**Answer:** Two mechanisms:
1. **Enable "Do not allow bypassing the above settings"** (GitHub calls this "Include administrators") in branch protection. Admins cannot merge without passing checks, even via the API.
2. **Rulesets** (GitHub Enterprise/Teams): Repository rulesets can be configured at the organization level by org owners and push down to all repos. Individual repo admins cannot override org-level rulesets. The org security team maintains the non-bypassable rules.

The principle: critical security controls should be enforced at a level *above* the person who might be tempted to bypass them.